In [ ]:
import pandas as pd
import re

In [ ]:
### define setting to indicate raw data in saved files
database = "MGnify" # MGnify / IGC / Qin
grouping = "Subgroups" # Groups / Subgroups
new_discovery_study = "Werner" # ValdesMas / Werner
file_prefix = database +"_"+grouping + "_" + new_discovery_study + "_"
print(file_prefix)

prophane_results_path = database + "_" + grouping + "(DiscoveryJavaOutput)/prophane_results/"

sc_threshold = 5

In [ ]:
### load metadata
meta_data_path = r"SupplementaryFile1(WernerDiscovery)-Revision.xlsx"
meta_data = pd.read_excel(meta_data_path, sheet_name="SampleMetadata", index_col = "SampleID" )
meta_data.columns = [re.sub(r'[^A-Za-z0-9]+', '_', col) for col in meta_data.columns]
print(meta_data.head(5))

# Process Prophane Output & Add EC, KO & GO:Term annotation

In [ ]:
prophaneSummary =pd.read_csv("./"+prophane_results_path+"summary.txt",sep="\t")
#print(prophaneSummary.head())
#print(prophaneSummary.columns)
#print(prophaneSummary["task_0::Functional_Annotation_Task_1::og"])
prophaneSummary.loc[prophaneSummary["level"]=="group"]["task_0::Functional_Annotation_Task_1::og"].value_counts()

### select "group" level, than use members_identifier to collect information about EC, KO, and GO:Terms
### search for KO and EC numbers in eggNOG annotation dataframe
proteinGroupAnnotation = prophaneSummary.loc[prophaneSummary["level"]=="group"].set_index("#pg")
#proteinGroupAnnotation.head(2)

### eggNOG result file --> lines starting with "#" have to be excluded, but line starting with "#query" holds column names
with open("./"+prophane_results_path+"tasks/fun_annot_by_emapper_v2_on_eggnog_v5.v5.0.2.task0.result") as file:
    for line in file:
        if line[0:2] =="#q":
            column_header= line.split("\t")
            print(column_header)

# read result file as pd dataframe, "comment = #" to exclude line starting with "#", set extracted column names as header (names = column_header) 
eggNOG_annotations = pd.read_csv("./"+prophane_results_path+ "tasks/fun_annot_by_emapper_v2_on_eggnog_v5.v5.0.2.task0.result",sep="\t",names=column_header, comment="#", low_memory=False).set_index("#query")
eggNOG_annotations.index


new_col = None
### create newe columns for Annotations
proteinGroupAnnotation.insert(loc = 19, column = 'KO', value = new_col)
proteinGroupAnnotation.insert(loc = 19, column = 'EC', value = new_col)
proteinGroupAnnotation.insert(loc = 19, column = 'GO:Term', value = new_col)
proteinGroupAnnotation.insert(loc = 19, column = 'Kegg_Reaction', value = new_col)

#proteinGroupAnnotation["KO"] = None
#proteinGroupAnnotation["EC"] = None
#proteinGroupAnnotation["GO:Term"] = None
#proteinGroupAnnotation["Kegg_Reaction"] = None

### extract annotation based on EggNOG OG
for proteinGroup in proteinGroupAnnotation.index: #proteinGroupAnnotation.index:
    print(proteinGroup)
    # collect all accessions for this group
    proteinAccessions = proteinGroupAnnotation.loc[proteinGroup, "members_identifier"]

    #'GOs', 'EC', 'KEGG_ko', "KEGG_Reaction"
    KO = []
    EC = []
    GO = []
    Kegg_reaction = []

    # search for annotations in EggNOG annotations:
    for protein in proteinAccessions.split(";"):
        if protein in eggNOG_annotations.index: # not all proteins in eggNOG results...reason?
            #print(protein)
            ec = eggNOG_annotations.loc[protein, "EC"].split(",")
            EC.extend(ec)
            #print(set(EC))
            ko = eggNOG_annotations.loc[protein, "KEGG_ko"].split(",")
            KO.extend(ko)
            go = eggNOG_annotations.at[protein, "GOs"].split(",")
            GO.extend(go)
            kr = eggNOG_annotations.loc[protein, "KEGG_Reaction"].split(",")
            Kegg_reaction.extend(kr)
        #print(GO)
    proteinGroupAnnotation.loc[proteinGroup,"KO"] = str(set(KO))
    proteinGroupAnnotation.loc[proteinGroup,"EC"] = str(set(EC))
    #print(set(GO))
    proteinGroupAnnotation.loc[proteinGroup,"GO:Term"] = str(set(GO))
    proteinGroupAnnotation.loc[proteinGroup,"Kegg_Reaction"] = str(set(Kegg_reaction))


In [ ]:
# Drop Columns not used from Prophane summary (raw_quant, standard deviation, annotation checks)
proteinGroupAnnotation = proteinGroupAnnotation[proteinGroupAnnotation.columns.drop(list(proteinGroupAnnotation.filter(regex='raw_quant')))]
proteinGroupAnnotation = proteinGroupAnnotation[proteinGroupAnnotation.columns.drop(list(proteinGroupAnnotation.filter(regex='quant_sd')))]
proteinGroupAnnotation = proteinGroupAnnotation.drop(proteinGroupAnnotation.columns[-12:],axis = "columns")

### Shorten and adapt column names for easier matching
# delete Mascot Batch Information and .mgf and .dat
proteinGroupAnnotation.columns = [re.sub(r".*___(.*)\\.mgf.*", '\\1', col) for col in proteinGroupAnnotation.columns]
# exchange all special characters for "_" 

proteinGroupAnnotation.columns = [re.sub(r'[^A-Za-z0-9]+', '_', col) for col in proteinGroupAnnotation.columns]

# save to prophane summary directory
proteinGroupAnnotation.to_csv("./"+prophane_results_path+file_prefix+"summary.csv")

# Normalization & Filtering

In [ ]:
proteinGroupAnnotation = pd.read_csv("./"+prophane_results_path+file_prefix+"summary.csv", index_col = "#pg")
proteinGroupAnnotation.columns[0:25]

In [ ]:
# Prophane Output
annotation_columns = proteinGroupAnnotation.columns[0:23]
compare_panel = proteinGroupAnnotation

compare_panel.drop(annotation_columns, axis=1, inplace = True)
compare_panel.head(5)

compare_panel.rename(columns=lambda s: s.replace("P36_UCa", "P35_UCa"), inplace=True)

### check wether samples have an unusal number of identified metaproteins
print(compare_panel.astype(bool).sum(axis=0).sort_values()[0:20])

### filter metaproteins for summed abundace of more than 10 Spectral counts
compare_panel["Sum of Spectal counts"]= compare_panel.sum(axis=1)
compare_panel = compare_panel[compare_panel['Sum of Spectal counts'] >= sc_threshold].drop([ "Sum of Spectal counts"], axis = "columns")
#compare_panel.head()

In [ ]:
### normalization to relative abundance
#compare_panel.sum(axis="rows")
compare_panel = compare_panel.div(compare_panel.sum(axis="rows"), axis="columns")
compare_panel.to_csv(file_prefix + "all_metaproteins(all_studies_normalized_"+str(sc_threshold)+").csv")

In [ ]:
### for discovery dataset, extract the shared metaproteins (metaproteins, where sum across samples in a study is >0)
### join sample annotation and metaproteni abundance
for measurementID in compare_panel.columns:
    #print(type(measurementID))
    for sampleID in meta_data.columns:
        if sampleID in measurementID:
            compare_panel.loc["study", measurementID]= meta_data.loc["study", sampleID]
            compare_panel.loc["UseCase", measurementID]= meta_data.loc["UseCase", sampleID]
print(compare_panel.columns)

### somewhere here, a sample is los "35_UC

### Angleichen von ID in MetaData File und Measurement File...


row = compare_panel.loc["UseCase"]

discovery_columns = row[row == 'BiomarkerDiscovery'].index
discovery_panel = compare_panel[discovery_columns]

n_studies = len(discovery_panel.loc["study"].unique())

for study in discovery_panel.loc["study"].unique():
    row = compare_panel.loc["study"]
    discovery_columns = row[row == study].index
    study_panel = compare_panel[discovery_columns]
    #print(study_panel.sum(axis="columns"))    
    discovery_panel[str("Sum (" + study + ")")]=study_panel.sum(axis="columns")
        
    ### sum value of each metaprotein in these samples
discovery_panel = discovery_panel.drop(["UseCase", "study"], axis = "rows")


### delete all rows where product the product of all sums in each study is 0 (--> only share metaproteins retained)
discovery_panel = discovery_panel[discovery_panel.iloc[:, -n_studies:].prod(axis=1)!=0]
discovery_panel = discovery_panel.iloc[:, :-n_studies]
discovery_panel.to_csv(file_prefix + "shared_metaproteins(discovery_studies_normalized_"+str(sc_threshold)+").csv")
print(len(discovery_panel.index))